# 04 – NLP Analysis

### Purpose of the Notebook
Analyse textbasierter Vergabefelder mittels NLP.

### Steps
- TF‑IDF preprocessing
- SVD dimensionality reduction
- NMF topic modelling
- SVM text‑risk classifier (3 classes)
- Export TEXT_RISK_SCORE + NLP features
- Integration into modelling pipeline

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

import pickle

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [25]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.nlp_processing import (text_preprocessing, build_tfidf_svd, build_nmf_topics,
                               build_text_risk_classifier, predict_text_risk, map_risk_from_offers)

from my_scripts.eda import (overview, filter_germany)
from my_scripts.feature_engineering import (create_topic_id, map_topic_name)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset_fe.pkl")
df = df.reset_index(drop=True)

print("EU dataset:", df.shape)

EU dataset: (3518937, 15)


----------------
### NLP PROCESSING

----------

In [7]:
# ---------------------------------------------------------
# Preprocess text 
# ---------------------------------------------------------

df = text_preprocessing(df, ["TITLE"])


In [8]:
df.shape

(3518937, 15)

In [9]:
# ---------------------------------------------------------
# TF‑IDF + SVD features
# ---------------------------------------------------------

df_svd, X_tfidf, tfidf_vectorizer, svd_model = build_tfidf_svd(df["TITLE"])
df = pd.concat([df, df_svd], axis=1)


In [10]:
# ---------------------------------------------------------
# Topic modelling (NMF)
# ---------------------------------------------------------
df_topics, nmf_model = build_nmf_topics(X_tfidf)
df = pd.concat([df, df_topics], axis=1)


In [11]:
df.shape

(3518937, 130)

In [12]:
# ---------------------------------------------------------
# Labels for TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_LABEL"] = df["NUMBER_OFFERS"].apply(map_risk_from_offers)


In [14]:
# ---------------------------------------------------------
# Train SVM classifier
# ---------------------------------------------------------

df_text = df[df["TEXT_RISK_LABEL"].notna()].copy()

texts = df_text["TITLE"].fillna("").astype(str)
labels = df_text["TEXT_RISK_LABEL"].astype(str)

svm_model, tfidf_svm, label_encoder = build_text_risk_classifier(texts, labels)



In [15]:
# ---------------------------------------------------------
# Predict TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_SCORE"] = predict_text_risk(
    df["TITLE"],
    svm_model,
    tfidf_svm,
    label_encoder
)



In [16]:
df.head()

,YEAR,ISO_COUNTRY_CODE,TYPE_OF_CONTRACT,TOP_TYPE,VALUE_EURO,CRIT_PRICE_WEIGHT,NUMBER_OFFERS,TITLE,IS_FAILED_TENDER,AWARD_QUARTER,DAYS_TO_AWARD,VALUE_EURO_MISSING,CRIT_PRICE_WEIGHT_MISSING,CPV_CATEGORY,CAE_TYPE_CATEGORY,NLP_SVD_0,NLP_SVD_1,NLP_SVD_2,NLP_SVD_3,NLP_SVD_4,NLP_SVD_5,NLP_SVD_6,NLP_SVD_7,NLP_SVD_8,NLP_SVD_9,NLP_SVD_10,NLP_SVD_11,NLP_SVD_12,NLP_SVD_13,NLP_SVD_14,NLP_SVD_15,NLP_SVD_16,NLP_SVD_17,NLP_SVD_18,NLP_SVD_19,NLP_SVD_20,NLP_SVD_21,NLP_SVD_22,NLP_SVD_23,NLP_SVD_24,NLP_SVD_25,NLP_SVD_26,NLP_SVD_27,NLP_SVD_28,NLP_SVD_29,NLP_SVD_30,NLP_SVD_31,NLP_SVD_32,NLP_SVD_33,NLP_SVD_34,NLP_SVD_35,NLP_SVD_36,NLP_SVD_37,NLP_SVD_38,NLP_SVD_39,NLP_SVD_40,NLP_SVD_41,NLP_SVD_42,NLP_SVD_43,NLP_SVD_44,NLP_SVD_45,NLP_SVD_46,NLP_SVD_47,NLP_SVD_48,NLP_SVD_49,NLP_SVD_50,NLP_SVD_51,NLP_SVD_52,NLP_SVD_53,NLP_SVD_54,NLP_SVD_55,NLP_SVD_56,NLP_SVD_57,NLP_SVD_58,NLP_SVD_59,NLP_SVD_60,NLP_SVD_61,NLP_SVD_62,NLP_SVD_63,NLP_SVD_64,NLP_SVD_65,NLP_SVD_66,NLP_SVD_67,NLP_SVD_68,NLP_SVD_69,NLP_SVD_70,NLP_SVD_71,NLP_SVD_72,NLP_SVD_73,NLP_SVD_74,NLP_SVD_75,NLP_SVD_76,NLP_SVD_77,NLP_SVD_78,NLP_SVD_79,NLP_SVD_80,NLP_SVD_81,NLP_SVD_82,NLP_SVD_83,NLP_SVD_84,NLP_SVD_85,NLP_SVD_86,NLP_SVD_87,NLP_SVD_88,NLP_SVD_89,NLP_SVD_90,NLP_SVD_91,NLP_SVD_92,NLP_SVD_93,NLP_SVD_94,NLP_SVD_95,NLP_SVD_96,NLP_SVD_97,NLP_SVD_98,NLP_SVD_99,NLP_TOPIC_0,NLP_TOPIC_1,NLP_TOPIC_2,NLP_TOPIC_3,NLP_TOPIC_4,NLP_TOPIC_5,NLP_TOPIC_6,NLP_TOPIC_7,NLP_TOPIC_8,NLP_TOPIC_9,NLP_TOPIC_10,NLP_TOPIC_11,NLP_TOPIC_12,NLP_TOPIC_13,NLP_TOPIC_14,TEXT_RISK_LABEL,TEXT_RISK_SCORE
0,2008,DE,W,OPE,"512,402.17",100.00,2.00,,0,3.00,73.00,1,1,Other,Public Undertaking,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,medium,low
1,2008,DE,W,OPE,"512,402.17",100.00,3.00,,0,4.00,6.00,1,0,Other,National Agency,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,medium,low
2,2008,FR,W,OPE,"512,402.17",100.00,1.00,,1,4.00,37.00,1,1,Other,National Agency,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,high,low
3,2008,ES,W,OPE,"512,402.17",100.00,6.00,,0,4.00,63.00,1,1,Other,Ministry,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00

In [17]:
df.shape

(3518937, 132)

In [18]:
df.columns

Index(['YEAR', 'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'TOP_TYPE',
       'VALUE_EURO', 'CRIT_PRICE_WEIGHT', 'NUMBER_OFFERS', 'TITLE',
       'IS_FAILED_TENDER', 'AWARD_QUARTER',
       ...
       'NLP_TOPIC_7', 'NLP_TOPIC_8', 'NLP_TOPIC_9', 'NLP_TOPIC_10',
       'NLP_TOPIC_11', 'NLP_TOPIC_12', 'NLP_TOPIC_13', 'NLP_TOPIC_14',
       'TEXT_RISK_LABEL', 'TEXT_RISK_SCORE'],
      dtype='str', length=132)

#### Notes: NLP Pipeline for Tender Risk Prediction

1. Dataset Size After Features Engineering
- Full EU dataset:
  - Before: 3,518,937 rows × 15 columns
  - After: 3,518,937 rows × 132 columns

2. Text fields provide additional signals not captured by structured variables.  
Short titles and evaluation‑related text (TITLE) reveal complexity, niche requirements, and multi‑criteria scoring patterns that strongly influence bidder participation and failure risk.

3. TF‑IDF offers a scalable and domain‑appropriate representation of tender text.  
It efficiently captures important terms and patterns without requiring heavy linguistic models, making it suitable for millions of records and classical ML workflows.

4. Dimensionality reduction (SVD) converts high‑dimensional TF‑IDF vectors into compact numerical features.  
This reduces sparsity, stabilizes downstream models, and enables seamless integration with structured predictors such as CPV, procedure type, and tender value.

5. Topic modelling (NMF) introduces interpretable thematic structure.  
Extracted topics highlight procurement areas with systematically higher failure rates, improving interpretability and analytical insight.

6. A text‑based classifier (TF‑IDF + SVM) produces a high‑level TEXT_RISK_SCORE.  
It learns patterns associated with failed, medium‑risk, and safe tenders based solely on text, generating a categorical risk signal usable even when raw text is unavailable.

7. TEXT_RISK_SCORE is essential for downstream applications such as the risk simulator.  
It allows the model to incorporate text‑derived risk information without requiring free‑text input, enabling scenario simulations based on a small set of structured parameters.

8. The combined pipeline remains interpretable, scalable, and robust.  
TF‑IDF + SVD ensures numerical stability, NMF adds thematic insight, and SVM provides a practical risk score — together forming a balanced NLP module that strengthens the overall tender risk prediction model.

--------------
## Top topics

-------------

In [19]:
# ---------------------------------------------------------
# General Topics
# ---------------------------------------------------------

#choice topics

feature_names = tfidf_vectorizer.get_feature_names_out()

topics = {}

for i, topic in enumerate(nmf_model.components_):
    top_indices = topic.argsort()[-20:]  # топ 20 слів
    top_words = [feature_names[j] for j in top_indices]
    topics[f"Topic_{i}"] = top_words


In [20]:
# print topics
for t, words in topics.items():
    print(f"{t}: {', '.join(words)}")


Topic_0: nr 23, nr 24, nr 22, nr 21, nr 19, nr 20, nr 18, nr 17, pozycja, nr 16, nr poz, nr 15, nr 14, nr 13, nr 12, nr 11, nr 10, pakiet, nr, pakiet nr
Topic_1: leki zad, zad, lne, og lne, og, leki og, leki stosowane, stosowane, onkologiczne, leki onkologiczne, leki psychotropowe, psychotropowe, ce, pakiet leki, ii, nr leki, leki ii, ne, leki ne, leki
Topic_2: del, suministro, servicio, prestations de, travaux de, le, sur, les, prestations, travaux, du, para, en, et de, pour, fourniture de, fourniture, de la, la, de
Topic_3: 19, nr 13, 18, 17, nr 12, 16, poz, pakiet zadanie, nr 11, 15, 14, 13, nr 10, nr dostawa, 12, 11, 10, nr, zadanie nr, zadanie
Topic_4: pakiet 14, pakiet 13, pakiet 12, 19, pakiet 11, 24, 18, 16, pakiet 10, 15, 17, 14, 12, 13, 11, 10, pakiet poz, pakiet nr, poz, pakiet
Topic_5: 15, grupa 20, grupa 18, 16, ii, grupa 17, 14, grupa 15, grupa 16, 13, 12, grupa 14, grupa 13, grupa 12, 11, grupa 11, 10, grupa 10, za, grupa
Topic_6: tu, dostawa wyrob, jednorazowego ytku, y

#### Notes: Interpreted Topic Categories
- Topic 0 — Lot / Position Structure
- Topic 1 — Pharmaceuticals (General, Oncology, Psychotropic)
- Topic 2 — Works, Services and Supplies (FR/ES/PT)
- Topic 3 — Tasks and Lots (Structured Items)
- Topic 4 — Procurement Packages
- Topic 5 — Product / Service Groups
- Topic 6 — Medical Devices and Pharmaceuticals
- Topic 7 — Lot Selection (FR)
- Topic 8 — Public Utility / Municipal Services
- Topic 9 — Cardiovascular & Neurological Medicines
- Topic 10 — Procurement Parts / Segments (Part I/II/III)
- Topic 11 — Equipment, Services, Framework Agreements
- Topic 12 — Laboratory Materials and Reagents
- Topic 13 — Maintenance, Risks, Technical Services (FR)
- Topic 14 — Technical / IT / System Services (IT/DE/FR/IT)


In [21]:
# ---------------------------------------------------------
# Top Topics for Germany
# ---------------------------------------------------------

# Germany dataset
df_de = filter_germany(df)

# topics range for Germany

topic_cols = [c for c in df_de.columns if c.startswith("NLP_TOPIC_")]
df_de["dominant_topic"] = df_de[topic_cols].idxmax(axis=1)
df_de["dominant_topic"].value_counts()

dominant_topic
NLP_TOPIC_0     169977
NLP_TOPIC_14     74069
NLP_TOPIC_13      4119
NLP_TOPIC_12      3703
NLP_TOPIC_11      2702
NLP_TOPIC_10      1292
NLP_TOPIC_5        436
NLP_TOPIC_7        405
NLP_TOPIC_2        246
NLP_TOPIC_6        245
NLP_TOPIC_8        186
NLP_TOPIC_9         43
NLP_TOPIC_3         22
Name: count, dtype: int64

### Germany‑Specific Topic Profile (based on PCA, KMeans, and dominant topic distribution)
1. Tender Lots & Procurement Packages (Topic 0)
2. Technical Criteria (Topic 6)
3. Pharmaceuticals & Medical Drugs (Topic 9)
4. Service & Management Contracts (Topic 10)
5. Technical Services & Performance Criteria (Topic 12)
6. Price–Quality Scoring (IT/FR patterns) (Topic 8)
7. Scoring Formulas (price/quality ratios) (Topic 7)
8. Lot Numbering & Sub‑lot Structure (Topic 2)
9. General Evaluation Criteria (Topic 14)
10. Quality–Price Evaluation (FR/IT) (Topic 11)

# topic feature

In [26]:
df = create_topic_id(df)

In [27]:
df = map_topic_name(df)

In [28]:
df.head()

,YEAR,ISO_COUNTRY_CODE,TYPE_OF_CONTRACT,TOP_TYPE,VALUE_EURO,CRIT_PRICE_WEIGHT,NUMBER_OFFERS,TITLE,IS_FAILED_TENDER,AWARD_QUARTER,DAYS_TO_AWARD,VALUE_EURO_MISSING,CRIT_PRICE_WEIGHT_MISSING,CPV_CATEGORY,CAE_TYPE_CATEGORY,NLP_SVD_0,NLP_SVD_1,NLP_SVD_2,NLP_SVD_3,NLP_SVD_4,NLP_SVD_5,NLP_SVD_6,NLP_SVD_7,NLP_SVD_8,NLP_SVD_9,NLP_SVD_10,NLP_SVD_11,NLP_SVD_12,NLP_SVD_13,NLP_SVD_14,NLP_SVD_15,NLP_SVD_16,NLP_SVD_17,NLP_SVD_18,NLP_SVD_19,NLP_SVD_20,NLP_SVD_21,NLP_SVD_22,NLP_SVD_23,NLP_SVD_24,NLP_SVD_25,NLP_SVD_26,NLP_SVD_27,NLP_SVD_28,NLP_SVD_29,NLP_SVD_30,NLP_SVD_31,NLP_SVD_32,NLP_SVD_33,NLP_SVD_34,NLP_SVD_35,NLP_SVD_36,NLP_SVD_37,NLP_SVD_38,NLP_SVD_39,NLP_SVD_40,NLP_SVD_41,NLP_SVD_42,NLP_SVD_43,NLP_SVD_44,NLP_SVD_45,NLP_SVD_46,NLP_SVD_47,NLP_SVD_48,NLP_SVD_49,NLP_SVD_50,NLP_SVD_51,NLP_SVD_52,NLP_SVD_53,NLP_SVD_54,NLP_SVD_55,NLP_SVD_56,NLP_SVD_57,NLP_SVD_58,NLP_SVD_59,NLP_SVD_60,NLP_SVD_61,NLP_SVD_62,NLP_SVD_63,NLP_SVD_64,NLP_SVD_65,NLP_SVD_66,NLP_SVD_67,NLP_SVD_68,NLP_SVD_69,NLP_SVD_70,NLP_SVD_71,NLP_SVD_72,NLP_SVD_73,NLP_SVD_74,NLP_SVD_75,NLP_SVD_76,NLP_SVD_77,NLP_SVD_78,NLP_SVD_79,NLP_SVD_80,NLP_SVD_81,NLP_SVD_82,NLP_SVD_83,NLP_SVD_84,NLP_SVD_85,NLP_SVD_86,NLP_SVD_87,NLP_SVD_88,NLP_SVD_89,NLP_SVD_90,NLP_SVD_91,NLP_SVD_92,NLP_SVD_93,NLP_SVD_94,NLP_SVD_95,NLP_SVD_96,NLP_SVD_97,NLP_SVD_98,NLP_SVD_99,NLP_TOPIC_0,NLP_TOPIC_1,NLP_TOPIC_2,NLP_TOPIC_3,NLP_TOPIC_4,NLP_TOPIC_5,NLP_TOPIC_6,NLP_TOPIC_7,NLP_TOPIC_8,NLP_TOPIC_9,NLP_TOPIC_10,NLP_TOPIC_11,NLP_TOPIC_12,NLP_TOPIC_13,NLP_TOPIC_14,TEXT_RISK_LABEL,TEXT_RISK_SCORE,TOPIC_ID,TOPIC_NAME
0,2008,DE,W,OPE,"512,402.17",100.00,2.00,,0,3.00,73.00,1,1,Other,Public Undertaking,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,medium,low,0,Lot / Position Structure
1,2008,DE,W,OPE,"512,402.17",100.00,3.00,,0,4.00,6.00,1,0,Other,National Agency,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,medium,low,0,Lot / Position Structure
2,2008,FR,W,OPE,"512,402.17",100.00,1.00,,1,4.00,37.00,1,1,Other,National Agency,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,high,low,0,Lot / Position Structure
3,2008,ES,W,OPE,"512,402.17",100.00,6.00,,0,4.00,63.00,1,1,Other,Ministry,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0

--------------
### SAVE DATASET & MODEL

--------------

In [29]:
# Topics for Germany 

df_de = filter_germany(df)

topic_cols = [c for c in df_de.columns if c.startswith("NLP_TOPIC_")]

df_de_topics = df_de[topic_cols].copy()

df_de_topics.to_pickle("../data/dataset_topics_de.pkl")

In [32]:
# save dataset
df.to_pickle("../data/dataset_nlp.pkl")

In [33]:
# save modell

with open("../models/nmf_model.pkl", "wb") as f:
    pickle.dump(nmf_model, f)

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)
